# Aktivasyon Fonksiyonlarının Eğitim Dinamiklerine Etkisi
## Effect of Activation Functions on Training Dynamics

**Ders:** SWE012 – Deep Learning with Python  
**Konu:** Aktivasyon fonksiyonlarının (Sigmoid, Tanh, ReLU, Leaky ReLU) eğitim süreci üzerindeki etkilerinin kontrollü deneylerle incelenmesi.  

**Yaklaşım:**
- **Derinlik:** Aktivasyon fonksiyonları üzerine sistematik deneyler
- **Genişlik:** Derste işlenen tüm metodolojilerin (ML temelleri, regularization, optimization, initialization) aktivasyon fonksiyonlarıyla etkileşimi

---

## 1. Kütüphaneler ve Kurulum

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Veri Seti: Fashion-MNIST

Fashion-MNIST, 10 sınıflı bir görüntü sınıflandırma problemidir (28×28 gri tonlamalı).  
- **i.i.d. varsayımı:** Train ve test setleri aynı dağılımdan bağımsız örneklenmiştir.  
- **Generalization:** Modelin train'de değil, **test** setindeki performansı asıl ölçümüzdür.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

BATCH_SIZE = 128  # Tipik minibatch boyutu (SGD noise için yeterli)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"Sınıf sayısı: {len(class_names)} | Giriş boyutu: 28x28 = 784")
print(f"Minibatch boyutu: {BATCH_SIZE} -> Epoch başına ~{len(train_loader)} iterasyon")

In [ ]:
# Örnek görseller
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')
plt.suptitle('Fashion-MNIST Örnekleri', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Model Mimarisi ve Eğitim Altyapısı

**Feedforward Neural Network:** Giriş → Gizli Katmanlar → Softmax Çıkış  
- **Softmax + CrossEntropyLoss:** 10 sınıflı sınıflandırma için standart eşleşme (logit'ler doğrudan verilir, softmax loss içinde)  
- **MLE bağlantısı:** Cross-entropy loss minimize etmek = Categorical dağılım altında MLE yapmak  
- **Backpropagation:** Chain rule + memoization ile gradyanlar hesaplanır

In [ ]:
ACTIVATION_FUNCTIONS = {
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
    'ReLU': nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(0.01),
}

def build_model(activation_name, hidden_sizes=[256, 128], use_dropout=False, 
                use_batchnorm=False, input_size=784, num_classes=10):
    """Kontrollü deney için parametrik model oluşturma."""
    layers = []
    in_features = input_size
    
    for h in hidden_sizes:
        layers.append(nn.Linear(in_features, h))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(h))  # BN: normalize -> scale(gamma) + shift(beta)
        layers.append(ACTIVATION_FUNCTIONS[activation_name].__class__() 
                      if activation_name != 'LeakyReLU' 
                      else nn.LeakyReLU(0.01))
        if use_dropout:
            layers.append(nn.Dropout(0.5))  # p=0.5: drop probability (hidden layers)
        in_features = h
    
    layers.append(nn.Linear(in_features, num_classes))  # Çıkış: raw logits (softmax loss içinde)
    return nn.Sequential(*layers)


def init_weights(model, method='he'):
    """Ağırlık başlatma stratejileri."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if method == 'he':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')  # He: Var = 2/n_in
            elif method == 'xavier':
                nn.init.xavier_normal_(m.weight)  # Xavier: Var = 2/(n_in + n_out)
            elif method == 'random':
                nn.init.normal_(m.weight, 0, 0.5)  # Büyük varyans -> exploding risk
            nn.init.zeros_(m.bias)
    return model

In [ ]:
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs=15):
    """Eğitim döngüsü: train loss, test loss, test accuracy kayıtları."""
    model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)         # Forward pass
            loss = criterion(logits, y_batch) # Loss hesaplama
            loss.backward()                  # Backward pass (backpropagation)
            optimizer.step()                 # Parametre güncelleme
            running_loss += loss.item()
        
        train_loss = running_loss / len(train_loader)
        
        # --- Evaluation ---
        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                test_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        
        test_loss /= len(test_loader)
        test_acc = correct / total
        
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
    
    return history

In [ ]:
def plot_results(results_dict, title, ylabel='Loss'):
    """Deney sonuçlarını görselleştirme."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    for name, hist in results_dict.items():
        axes[0].plot(hist['train_loss'], label=name)
        axes[1].plot(hist['test_loss'], label=name)
        axes[2].plot(hist['test_acc'], label=name)
    
    axes[0].set_title(f'{title} — Train Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].set_title(f'{title} — Test Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()
    
    axes[2].set_title(f'{title} — Test Accuracy')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()


def summary_table(results_dict):
    """Sonuç özet tablosu."""
    print(f"{'Konfigürasyon':<30} {'Son Train Loss':>15} {'Son Test Loss':>15} {'En İyi Test Acc':>17}")
    print('-' * 80)
    for name, hist in results_dict.items():
        best_acc = max(hist['test_acc'])
        print(f"{name:<30} {hist['train_loss'][-1]:>15.4f} {hist['test_loss'][-1]:>15.4f} {best_acc:>16.4f}")

---
## 4. DENEY 1: Aktivasyon Fonksiyonu Karşılaştırması (Derinlik)

**Kontrollü Deney Tasarımı:** Tüm parametreler sabit, yalnızca aktivasyon fonksiyonu değişiyor.  
- Mimari: 784 → 256 → 128 → 10  
- Optimizer: Adam (lr=0.001) — endüstri standardı (Momentum + RMSProp + bias correction)  
- Initialization: He (Kaiming) — ReLU ailesine uygun  
- Regularization: Yok (saf aktivasyon etkisini görmek için)  
- Epoch: 15  

**Beklenti:** ReLU ve Leaky ReLU, Sigmoid ve Tanh'a göre daha hızlı yakınsayacak (vanishing gradient problemi nedeniyle).

In [ ]:
EPOCHS = 15
LR = 0.001

results_exp1 = {}
for act_name in ACTIVATION_FUNCTIONS:
    print(f"Eğitiliyor: {act_name}...", end=' ')
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()  # Softmax + NLL = CCE (10 sınıf)
    
    history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    results_exp1[act_name] = history
    print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp1, 'Deney 1: Aktivasyon Fonksiyonu Karşılaştırması')
summary_table(results_exp1)

**Analiz:**
- **Sigmoid/Tanh:** Doygunluk bölgelerinde gradyan ≈ 0 → vanishing gradient → yavaş öğrenme  
- **ReLU:** x > 0 için gradyan = 1 → hızlı yakınsama, ama **dead neuron** riski (x < 0 → gradyan = 0)  
- **Leaky ReLU:** x < 0 için küçük gradyan (0.01) → dead neuron sorununu hafifletir  

---
## 5. DENEY 2: Aktivasyon × Optimizer Etkileşimi (Genişlik: Optimizasyon)

Derste gördüğümüz optimizasyon yöntemlerinin aktivasyon fonksiyonlarıyla nasıl etkileştiğini inceleyelim.  

| Optimizer | Özellik |
|-----------|--------|
| SGD | Temel güncelleme: w -= lr × ∇J |
| SGD + Momentum | Hız birikimi: v = βv + ∇J, sallantıyı azaltır |
| RMSProp | Per-parametre lr: AdaGrad'ın düzeltilmiş hali |
| Adam | Momentum + RMSProp + bias correction |

In [ ]:
optimizers_config = {
    'SGD': lambda params: optim.SGD(params, lr=0.01),
    'SGD+Momentum': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'RMSProp': lambda params: optim.RMSprop(params, lr=0.001),
    'Adam': lambda params: optim.Adam(params, lr=0.001),
}

# En iyi ve en kötü aktivasyonu seçerek optimizer etkisini gösterelim
test_activations = ['Sigmoid', 'ReLU']

results_exp2 = {}
for act_name in test_activations:
    for opt_name, opt_fn in optimizers_config.items():
        key = f"{act_name} + {opt_name}"
        print(f"Eğitiliyor: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method='he')
        optimizer = opt_fn(model.parameters())
        criterion = nn.CrossEntropyLoss()
        
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
        results_exp2[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp2, 'Deney 2: Aktivasyon × Optimizer')
summary_table(results_exp2)

**Analiz:**
- **Sigmoid + SGD:** En kötü kombinasyon — vanishing gradient + sabit learning rate  
- **Sigmoid + Adam:** Adaptive lr, Sigmoid'in yavaşlığını kısmen telafi eder  
- **ReLU + herhangi bir optimizer:** Güçlü gradyan akışı sayesinde tüm optimizer'lar iyi çalışır  
- **Momentum etkisi:** SGD'ye momentum eklemek, özellikle Sigmoid gibi zayıf gradyanlı fonksiyonlarda büyük fark yaratır

---
## 6. DENEY 3: Aktivasyon × Regularizasyon (Genişlik: Regularization)

Overfitting'i kontrol altına almak için farklı regularizasyon tekniklerinin aktivasyon fonksiyonlarıyla etkileşimi.

| Yöntem | Mekanizma | Bayesian Yorumu |
|--------|-----------|----------------|
| L2 (Weight Decay) | ½α‖w‖² → ağırlıkları küçültür | Gaussian prior |
| L1 | α‖w‖₁ → seyreklik (sparse) | Laplace prior |
| Dropout (p=0.5) | Rastgele nöron kapatma → 2ⁿ alt ağ ensemble'ı | — |
| Batch Normalization | Her katmanda normalize → kararlı dağılım | — |
| Label Smoothing (ε=0.1) | Hedef: 1-ε doğru, ε/(k-1) yanlış sınıflar | — |

In [ ]:
# Label Smoothing ile CrossEntropy
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)  # epsilon = 0.1
criterion_normal = nn.CrossEntropyLoss()

reg_configs = {
    'No Reg':        {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'L2 (wd=1e-4)':  {'dropout': False, 'batchnorm': False, 'wd': 1e-4, 'criterion': criterion_normal},
    'Dropout(0.5)':  {'dropout': True,  'batchnorm': False, 'wd': 0,    'criterion': criterion_normal},
    'BatchNorm':     {'dropout': False, 'batchnorm': True,  'wd': 0,    'criterion': criterion_normal},
    'LabelSmooth':   {'dropout': False, 'batchnorm': False, 'wd': 0,    'criterion': criterion_smooth},
}

test_activations_reg = ['Sigmoid', 'ReLU']

results_exp3 = {}
for act_name in test_activations_reg:
    for reg_name, cfg in reg_configs.items():
        key = f"{act_name} + {reg_name}"
        print(f"Eğitiliyor: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name, use_dropout=cfg['dropout'], use_batchnorm=cfg['batchnorm'])
        model = init_weights(model, method='he')
        optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=cfg['wd'])
        
        history = train_model(model, train_loader, test_loader, optimizer, cfg['criterion'], EPOCHS)
        results_exp3[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp3, 'Deney 3: Aktivasyon × Regularizasyon')
summary_table(results_exp3)

**Analiz:**
- **Batch Normalization + Sigmoid/Tanh:** BN, internal covariate shift'i azaltarak Sigmoid/Tanh'ın doygunluk sorununu önemli ölçüde hafifletir  
- **Dropout:** Train loss'u yükseltir (öğrenmeyi zorlaştırır) ama generalization gap'i kapatır  
- **L2 vs L1:** L2 ağırlıkları küçültür ama sıfırlamaz; L1 gereksiz bağlantıları sıfırlayarak seyreklik sağlar  
- **Label Smoothing:** Modelin aşırı güvenli tahminler yapmasını engeller → daha iyi kalibre edilmiş olasılıklar

---
## 7. DENEY 4: Aktivasyon × Başlangıç Stratejisi (Genişlik: Initialization)

**Neden önemli?**
- Tüm ağırlıklar aynı → **simetri sorunu** (tüm nöronlar aynı şeyi öğrenir)  
- Ağırlıklar çok küçük → **vanishing gradients**  
- Ağırlıklar çok büyük → **exploding gradients**  

| Yöntem | Varyans | En Uygun Aktivasyon |
|--------|---------|--------------------|
| Xavier (Glorot) | 2/(n_in + n_out) | Sigmoid, Tanh |
| He (Kaiming) | 2/n_in | ReLU, Leaky ReLU |
| Random (σ=0.5) | Kontrolsüz | — (riskli) |

In [ ]:
init_methods = ['xavier', 'he', 'random']

results_exp4 = {}
for act_name in ACTIVATION_FUNCTIONS:
    for init_name in init_methods:
        key = f"{act_name} + {init_name}"
        print(f"Eğitiliyor: {key}...", end=' ')
        torch.manual_seed(42)
        model = build_model(act_name)
        model = init_weights(model, method=init_name)
        optimizer = optim.Adam(model.parameters(), lr=LR)
        criterion = nn.CrossEntropyLoss()
        
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
        results_exp4[key] = history
        print(f"Test Acc: {max(history['test_acc']):.4f}")

plot_results(results_exp4, 'Deney 4: Aktivasyon × Initialization')
summary_table(results_exp4)

**Analiz:**
- **Xavier + Sigmoid/Tanh:** Doğru eşleşme — Xavier, sinyalin sabit kalmasını sağlar  
- **He + ReLU/LeakyReLU:** Doğru eşleşme — He, ReLU'nun yarıya indirdiği varyansı 2× ile telafi eder  
- **Random:** Kontrolsüz varyans → özellikle derin ağlarda exploding/vanishing riski  

---
## 8. Gradyan Akış Analizi (Gradient Flow)

Backpropagation'da gradyanların katmanlar arası akışını doğrudan gözlemleyelim.  
Vanishing gradient = derin katmanlarda gradyan normu ≈ 0 → öğrenme durur.

In [ ]:
def get_gradient_norms(model, data_loader, criterion):
    """Tek batch için her katmanın gradyan normunu hesapla."""
    model.train()
    X_batch, y_batch = next(iter(data_loader))
    X_batch = X_batch.view(-1, 784).to(device)
    y_batch = y_batch.to(device)
    
    model.zero_grad()
    loss = criterion(model(X_batch), y_batch)
    loss.backward()
    
    grad_norms = []
    layer_names = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            grad_norms.append(param.grad.norm().item())
            layer_names.append(name)
    return layer_names, grad_norms

# Daha derin ağ ile gradient flow karşılaştırması
deep_hidden = [512, 256, 128, 64, 32]  # 5 gizli katman

fig, ax = plt.subplots(figsize=(12, 5))
for act_name in ACTIVATION_FUNCTIONS:
    torch.manual_seed(42)
    model = build_model(act_name, hidden_sizes=deep_hidden)
    model = init_weights(model, method='he')
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    names, norms = get_gradient_norms(model, train_loader, criterion)
    ax.plot(range(len(norms)), norms, 'o-', label=act_name, markersize=6)

ax.set_xlabel('Katman (girişe yakın → çıkışa yakın)')
ax.set_ylabel('Gradyan Normu')
ax.set_title('Gradyan Akışı: 5 Katmanlı Ağ (Başlangıç Durumu)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\nYorum: Sigmoid/Tanh'ta giriş katmanlarına doğru gradyan normu hızla düşer (vanishing gradient).")
print("ReLU/LeakyReLU'da gradyan akışı katmanlar boyunca çok daha stabil kalır.")

## 9. Ölü Nöron Analizi (Dead Neuron Problem)

ReLU'da x < 0 → çıkış = 0 ve gradyan = 0. Eğer bir nöronun girdisi sürekli negatif kalırsa, o nöron **ölür** ve bir daha öğrenemez. Leaky ReLU, küçük bir negatif eğim ile bunu önler.

In [ ]:
def count_dead_neurons(model, data_loader, num_batches=10):
    """ReLU/LeakyReLU sonrası sürekli 0 çıktı veren nöronları say."""
    model.eval()
    activations = {}  # layer_idx -> toplam aktivasyon
    counts = {}       # layer_idx -> batch sayısı
    
    hooks = []
    layer_idx = 0
    for module in model.modules():
        if isinstance(module, (nn.ReLU, nn.LeakyReLU)):
            idx = layer_idx
            def hook_fn(m, inp, out, idx=idx):
                act = (out > 0).float().mean(dim=0)  # Her nöron için aktif olma oranı
                if idx not in activations:
                    activations[idx] = act
                    counts[idx] = 1
                else:
                    activations[idx] += act
                    counts[idx] += 1
            hooks.append(module.register_forward_hook(hook_fn))
            layer_idx += 1
    
    with torch.no_grad():
        for i, (X_batch, _) in enumerate(data_loader):
            if i >= num_batches:
                break
            X_batch = X_batch.view(-1, 784).to(device)
            model(X_batch)
    
    for h in hooks:
        h.remove()
    
    results = {}
    for idx in activations:
        avg_act = activations[idx] / counts[idx]
        dead = (avg_act < 0.01).sum().item()  # %1'den az aktif = ölü
        total = avg_act.numel()
        results[f'Layer {idx}'] = (dead, total, dead/total*100)
    return results

# ReLU vs LeakyReLU ölü nöron karşılaştırması
for act_name in ['ReLU', 'LeakyReLU']:
    torch.manual_seed(42)
    model = build_model(act_name)
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    # Kısa eğitim
    train_model(model, train_loader, test_loader, optimizer, criterion, epochs=10)
    
    dead_info = count_dead_neurons(model, train_loader)
    print(f"\n{act_name} — Ölü Nöron Analizi (10 epoch sonrası):")
    for layer, (dead, total, pct) in dead_info.items():
        print(f"  {layer}: {dead}/{total} ölü nöron ({pct:.1f}%)")

---
## 10. Bias-Variance Perspektifi

Train loss ile test loss arasındaki **generalization gap**, modelin variance'ını (overfitting eğilimini) yansıtır.  
- **Yüksek bias** (underfitting): Hem train hem test loss yüksek → model kapasitesi yetersiz  
- **Yüksek variance** (overfitting): Train loss düşük, test loss yüksek → regularization gerekli  

Aktivasyon fonksiyonunun seçimi bu dengeyi doğrudan etkiler:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, act_name in enumerate(ACTIVATION_FUNCTIONS):
    hist = results_exp1[act_name]
    axes[i].plot(hist['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(hist['test_loss'], label='Test Loss', linewidth=2)
    axes[i].fill_between(range(EPOCHS), hist['train_loss'], hist['test_loss'], alpha=0.2, color='red')
    axes[i].set_title(f'{act_name}\nGen. Gap = {hist["test_loss"][-1] - hist["train_loss"][-1]:.3f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend(fontsize=8)

plt.suptitle('Bias-Variance: Generalization Gap (kırmızı alan = overfitting göstergesi)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 11. L1 vs L2 Regularizasyon Detaylı Karşılaştırma

- **L2 (Ridge):** J(w) = Loss + α/2 · ‖w‖² → ağırlıkları küçültür ama sıfırlamaz → Gaussian prior  
- **L1 (Lasso):** J(w) = Loss + α · ‖w‖₁ → ağırlıkları **tam sıfıra** iter → Laplace prior → feature selection  

In [ ]:
# L1 regularizasyon (PyTorch'ta manuel eklenmeli)
def train_with_l1(model, train_loader, test_loader, optimizer, criterion, l1_lambda, epochs=15):
    model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784).to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            # L1 penalty: alpha * sum(|w|)
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        train_loss = running_loss / len(train_loader)
        
        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784).to(device)
                y_batch = y_batch.to(device)
                logits = model(X_batch)
                test_loss += criterion(logits, y_batch).item()
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss / len(test_loader))
        history['test_acc'].append(correct / total)
    
    return history

# ReLU ile L1 vs L2 karşılaştırması
results_l1l2 = {}

for reg_type, config in [('No Reg', {'wd': 0, 'l1': 0}), 
                          ('L2 (1e-4)', {'wd': 1e-4, 'l1': 0}),
                          ('L1 (1e-5)', {'wd': 0, 'l1': 1e-5})]:
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    criterion = nn.CrossEntropyLoss()
    
    if config['l1'] > 0:
        history = train_with_l1(model, train_loader, test_loader, optimizer, criterion, config['l1'], EPOCHS)
    else:
        history = train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    results_l1l2[reg_type] = history

# Ağırlık dağılımı karşılaştırması
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (reg_type, config) in enumerate([('No Reg', {'wd': 0, 'l1': 0}), 
                                         ('L2', {'wd': 1e-4, 'l1': 0}),
                                         ('L1', {'wd': 0, 'l1': 1e-5})]):
    torch.manual_seed(42)
    model = build_model('ReLU')
    model = init_weights(model, method='he')
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=config['wd'])
    criterion = nn.CrossEntropyLoss()
    if config['l1'] > 0:
        train_with_l1(model, train_loader, test_loader, optimizer, criterion, config['l1'], EPOCHS)
    else:
        train_model(model, train_loader, test_loader, optimizer, criterion, EPOCHS)
    
    all_weights = torch.cat([p.data.cpu().flatten() for p in model.parameters()])
    axes[i].hist(all_weights.numpy(), bins=100, density=True, alpha=0.7)
    axes[i].set_title(f'{reg_type}\nSıfır ağırlık oranı: {(all_weights.abs() < 0.001).float().mean():.1%}')
    axes[i].set_xlabel('Ağırlık değeri')
    axes[i].set_xlim(-0.5, 0.5)

plt.suptitle('L1 vs L2: Ağırlık Dağılımları (L1 → seyreklik, L2 → küçültme)', fontsize=13)
plt.tight_layout()
plt.show()

summary_table(results_l1l2)

---
## 12. Sonuç Özet Tablosu ve Tartışma

In [ ]:
print("=" * 90)
print("GENEL SONUÇ ÖZETİ")
print("=" * 90)

print("\n[Deney 1] Aktivasyon Fonksiyonu Karşılaştırması (Derinlik):")
summary_table(results_exp1)

print("\n[Deney 2] Aktivasyon × Optimizer:")
summary_table(results_exp2)

print("\n[Deney 3] Aktivasyon × Regularizasyon:")
summary_table(results_exp3)

print("\n[Deney 4] Aktivasyon × Initialization:")
summary_table(results_exp4)

## Sonuçlar

### Derinlik: Aktivasyon Fonksiyonları
1. **ReLU/Leaky ReLU** hemen her konfigürasyonda Sigmoid/Tanh'a göre daha hızlı yakınsama ve daha yüksek doğruluk sağlar.
2. **Sigmoid** en yavaş öğrenme ve en düşük performansı gösterir (vanishing gradient).
3. **Leaky ReLU**, ReLU'nun dead neuron sorununu hafifletir.

### Genişlik: Diğer Metodolojiler
4. **Optimizer etkisi:** Adam, tüm aktivasyonlarda en stabil performansı verir. SGD, momentum olmadan Sigmoid ile çok zorlanır.
5. **Batch Normalization**, Sigmoid/Tanh ile birlikte kullanıldığında büyük performans artışı sağlar (internal covariate shift azalır).
6. **Initialization:** He init ReLU ile, Xavier init Sigmoid/Tanh ile doğru eşleşme sağlar.
7. **L1 → seyreklik** (feature selection); **L2 → genel küçültme** (tüm öznitelikler önemli olduğunda).
8. **Dropout** generalization gap'i kapatmada etkili ama eğitimi yavaşlatır.

### Pratik Öneri
- Varsayılan: **ReLU + He init + Adam + BatchNorm** → güvenli ve güçlü başlangıç noktası.
- Ölü nöron sorunu varsa: **Leaky ReLU** veya **ELU** deneyin.
- Overfitting varsa: **Dropout + L2** kombinasyonu etkili.

---
## Kapsanan Ders Konuları (Checklist)

| Hafta | Konu | Nerede Kullanıldı |
|-------|------|------------------|
| 2 | ML Temelleri: i.i.d., Generalization, Bias-Variance | Veri seti bölümü, Deney 1 analizi, Bölüm 10 |
| 2 | MLE ↔ Loss Function bağlantısı | Cross-entropy = Categorical MLE (Bölüm 3) |
| 2 | SGD ve Minibatch | Tüm eğitim döngüleri (batch_size=128) |
| 2 | Regularization temelleri (L1, L2) | Deney 3, Bölüm 11 (Bayesian yorum dahil) |
| 3-4 | Feedforward Networks, Depth vs Width | Model mimarisi (Bölüm 3) |
| 3-4 | Softmax, Cross-Entropy (BCE/CCE) | Loss function seçimi (CCE, 10 sınıf) |
| 3-4 | Backpropagation | Eğitim döngüsü, Gradyan akış analizi (Bölüm 8) |
| 3-4 | Aktivasyon Fonksiyonları (Sigmoid, Tanh, ReLU, Leaky ReLU) | **ANA KONU** — Tüm deneyler |
| 4 | L2 Weight Decay | Deney 3 |
| 4 | L1 Lasso (seyreklik) | Bölüm 11 |
| 4 | Dropout | Deney 3 |
| 4 | Label Smoothing | Deney 3 |
| 4 | Batch Normalization | Deney 3 |
| 5 | Initialization (Xavier, He) | Deney 4 |
| 5 | Optimizers (SGD, Momentum, RMSProp, Adam) | Deney 2 |
| 5 | Vanishing/Exploding Gradients | Gradyan akış analizi (Bölüm 8) |